# Taal Vista Hotel Web Scraping

## Purpose

This notebook collects publicly available information from permitted pages of the official Taal Vista Hotel website.

The workflow begins by checking website access rules and inspecting page structure. Data extraction will proceed gradually, beginning with accommodation information.

## Data Collection Principles

1. Collect only publicly accessible business information.

2. Respect website access rules and technical restrictions.

3. Use reasonable request delays.

4. Do not access the external booking system.

5. Do not collect unnecessary personal information.

6. Preserve source URLs and collection dates.

7. Save extracted information in `data/raw` without analytical cleaning.

In [1]:
from pathlib import Path
from datetime import date
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://www.taalvistahotel.com"
ROBOTS_URL = f"{BASE_URL}/robots.txt"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

robots_response = requests.get(
    ROBOTS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", robots_response.status_code)
print()
print(robots_response.text[:3000])

Status code: 200

User-agent: *
Crawl-Delay: 3



## Website Access Check

The website returned asuccessful response for `robots.txt`.

The published rules apply to all automated user agents and specify a crawl delay of three seconds. No disallowed paths were listed.

This project will therefore wait at least three seconds between page requests and will stop if the website returns an access restriction or rate limit response.

In [3]:
robot_parser = RobotFileParser()
robot_parser.set_url(ROBOTS_URL)
robot_parser.parse(robots_response.text.splitlines())

ROOMS_URL = f"{BASE_URL}/rooms/"

print("Rooms page permitted:", robot_parser.can_fetch(HEADERS["User-Agent"], ROOMS_URL))
print("Required crawl delay:", robot_parser.crawl_delay(HEADERS["User-Agent"]))

Rooms page permitted: True
Required crawl delay: 3


In [6]:
import time
import requests

BASE_URL = "https://www.taalvistahotel.com"
ROOMS_URL = f"{BASE_URL}/rooms/"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

time.sleep(3)

rooms_response = requests.get(
    ROOMS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", rooms_response.status_code)
print("Final URL:", rooms_response.url)
print("Content type:", rooms_response.headers.get("Content-Type"))
print("HTML characters:", len(rooms_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/rooms/
Content type: text/html; charset=UTF-8
HTML characters: 999862


## Rooms Page Access Result

The rooms page returned a successful HTTP status code of 200 and HTML content.

The page will first be inspected before extraction. This helps identify the correct HTML elements and prevents the scraper from collecting navigation, footer, script, and unrelated page content.

In [7]:
rooms_soup = BeautifulSoup(
    rooms_response.text,
    "lxml"
)

page_title = rooms_soup.title.get_text(
    strip=True
) if rooms_soup.title else "No title found"

print("Page title:", page_title)

Page title: ROOMS - Taal Vista Hotel


In [8]:
page_headings = []

for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text and heading_text not in page_headings:
        page_headings.append(heading_text)

print("Unique headings found:", len(page_headings))
print()

for heading_text in page_headings[:50]:
    print(heading_text)

Unique headings found: 15

Rooms
DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


## Initial Rooms Page Findings

The heading inspection identified possible room and suite names together with unrelated website content.

The extraction will not rely only on heading text. The surrounding HTML structure must be inspected to determine whether each room has a description, wing, bed type, view, capacity, room size, or amenities.

In [9]:
excluded_headings = [
    "Rooms",
    "Privacy Overview",
    "Chinese New Year Offerings at Taal Vista Hotel",
    "Breathe Fresh"
]

room_headings = []

for heading_text in page_headings:
    if heading_text not in excluded_headings:
        room_headings.append(heading_text)

print("Possible room headings:", len(room_headings))
print()

for room_heading in room_headings:
    print(room_heading)

Possible room headings: 11

DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE


In [10]:
for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text in room_headings:
        print("Room:", heading_text)
        print("Heading tag:", heading.name)
        print("Heading class:", heading.get("class"))
        print("Parent tag:", heading.parent.name)
        print("Parent class:", heading.parent.get("class"))
        print("=" * 50)

Room: DELUXE ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Room: PREMIER QUEEN ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-5629', 'style-local-115-c35']
Room: Two-Bedroom Deluxe Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c44']
Room: Batangas Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c49']
Room: superior room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c60']
Room: deluxe room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c64']
Room: ridge room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c7

In [11]:
sample_room_names = [
    "DELUXE ROOM",
    "PREMIER QUEEN ROOM"
]

for sample_room_name in sample_room_names:
    sample_heading = rooms_soup.find(
        "h3",
        string=lambda text: text and text.strip() == sample_room_name
    )

    print("ROOM:", sample_room_name)

    for level, ancestor in enumerate(sample_heading.parents):
        if level >= 7:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:400])

    print()
    print("=" * 70)

ROOM: DELUXE ROOM

Ancestor level: 0
Tag: div
Class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Text preview: DELUXE ROOM

Ancestor level: 1
Tag: div
Class: ['h-global-transition-all', 'h-heading', 'style-1580', 'style-local-115-c30', 'position-relative', 'h-element']
Text preview: DELUXE ROOM

Ancestor level: 2
Tag: div
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-100', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: DELUXE ROOM Stay in our newly renovated Deluxe Rooms, thoughtfully designed for quiet and intimate accommodation. Located in the Lake Wing, guests may choose between a King, Twin, or Queen bed to suit their preference. Each room
                                      offers views of Aguinaldo Highway or the hotel’s charming courtyard, with easy access to our exclusive gate to Skyranch for added lei

Ancestor level: 3
Tag: div
Class: ['d-flex', 'h-flex-basis', 'h-column__inner', 'h-px-

## Initial Room Record Extraction

The repeated `h-column__content` container holds each room heading and its associated descriptive content.

The first extraction will preserve the room names and descriptions as published. Capitalization, category standardization, and feature extraction will be handled later during data cleaning.

In [12]:
room_records = []

for heading in rooms_soup.find_all("h3"):
    room_name = heading.get_text(" ", strip=True)

    if room_name in room_headings:
        room_container = heading.find_parent(
            "div",
            class_="h-column__content"
        )

        paragraph_texts = []

        if room_container:
            for paragraph in room_container.find_all("p"):
                paragraph_text = paragraph.get_text(" ", strip=True)

                if paragraph_text:
                    paragraph_texts.append(paragraph_text)

        room_description = " ".join(paragraph_texts)

        room_records.append({
            "room_name": room_name,
            "room_description": room_description,
            "source_url": ROOMS_URL,
            "date_collected": date.today().isoformat()
        })

rooms_raw_df = pd.DataFrame(room_records)

print("Rows:", rooms_raw_df.shape[0])
print("Columns:", rooms_raw_df.shape[1])
print()
print("Missing values:")
print(rooms_raw_df.isnull().sum())

display(
    rooms_raw_df[
        ["room_name", "room_description"]
    ]
)

Rows: 11
Columns: 4

Missing values:
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64


,room_name,room_description
0,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou..."
1,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...
2,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ..."
3,Batangas Suite,Celebrate life’s milestones or simply unwind i...
4,superior room,Surrounded by the refreshing view of lush gree...
5,deluxe room,"Located on a higher floor, the rooms are taste..."
6,ridge room,Inspired by the rich culture and relaxing outd...
7,PREMIER ROOM,"For a more captivating and serene stay, our Pr..."
8,ONE-BEDROOM DELUXE SUITE,Features a spacious living room and the privac...
9,TAAL SUITE,Features a modern and fresh vibe with floor-to...


## Raw Room Data Quality Check

The initial extraction returned 11 room records with no missing descriptions.

Room names will be checked without considering capitalization. A repeated normalized name will not automatically be treated as a duplicate because the records may represent distinct products in different hotel wings.

In [13]:
rooms_raw_df.insert(
    0,
    "room_record_id",
    range(1, len(rooms_raw_df) + 1)
)

rooms_raw_df["description_length"] = (
    rooms_raw_df["room_description"].str.len()
)

rooms_raw_df["normalized_room_name"] = (
    rooms_raw_df["room_name"]
    .str.strip()
    .str.lower()
)

duplicate_name_check = rooms_raw_df[
    rooms_raw_df.duplicated(
        subset="normalized_room_name",
        keep=False
    )
]

print("Exact duplicate rows:", rooms_raw_df.duplicated().sum())
print(
    "Repeated normalized room names:",
    duplicate_name_check.shape[0]
)
print()

display(
    duplicate_name_check[
        [
            "room_record_id",
            "room_name",
            "description_length",
            "room_description"
        ]
    ]
)

Exact duplicate rows: 0
Repeated normalized room names: 2



,room_record_id,room_name,description_length,room_description
0,1,DELUXE ROOM,789,"Stay in our newly renovated Deluxe Rooms, thou..."
5,6,deluxe room,336,"Located on a higher floor, the rooms are taste..."


## Raw Data Export Decision

The two Deluxe Room records will be preserved because they have different descriptions and may represent distinct room products.

Helper columns created for quality checking will not be included in the raw export. The raw dataset will contain only the record identifier, original room name, original description, source URL, and collection date.

In [14]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_data_folder = project_root / "data" / "raw"
raw_data_folder.mkdir(parents=True, exist_ok=True)

rooms_export_df = rooms_raw_df[
    [
        "room_record_id",
        "room_name",
        "room_description",
        "source_url",
        "date_collected"
    ]
].copy()

rooms_file = raw_data_folder / "taal_vista_rooms_raw.csv"

rooms_export_df.to_csv(
    rooms_file,
    index=False
)

print("File saved:", rooms_file)
print("Rows saved:", rooms_export_df.shape[0])
print("Columns saved:", rooms_export_df.shape[1])
print("File exists:", rooms_file.exists())

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_rooms_raw.csv
Rows saved: 11
Columns saved: 5
File exists: True


## Raw CSV Verification

The exported CSV will be loaded again to confirm that it is readable and that its dimensions, columns, missing values, and duplicate records remain correct after export.

In [15]:
rooms_verification_df = pd.read_csv(rooms_file)

print("Loaded shape:", rooms_verification_df.shape)
print()
print("Columns:")
print(rooms_verification_df.columns.tolist())
print()
print("Missing values:")
print(rooms_verification_df.isnull().sum())
print()
print(
    "Exact duplicate records:",
    rooms_verification_df.duplicated().sum()
)

display(rooms_verification_df.head())

Loaded shape: (11, 5)

Columns:
['room_record_id', 'room_name', 'room_description', 'source_url', 'date_collected']

Missing values:
room_record_id      0
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64

Exact duplicate records: 0


,room_record_id,room_name,room_description,source_url,date_collected
0,1,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou...",https://www.taalvistahotel.com/rooms/,2026-09-02
1,2,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...,https://www.taalvistahotel.com/rooms/,2026-09-02
2,3,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ...",https://www.taalvistahotel.com/rooms/,2026-09-02
3,4,Batangas Suite,Celebrate life’s milestones or simply unwind i...,https://www.taalvistahotel.com/rooms/,2026-09-02
4,5,superior room,Surrounded by the refreshing view of lush gree...,https://www.taalvistahotel.com/rooms/,2026-09-02


# Dining Data Collection

## Purpose

This section checks the official dining page and identifies publicly presented restaurants, bars, cafés, operating schedules, cuisines, and descriptions.

The page structure will be inspected before any dining records are extracted.

In [16]:
DINING_URL = f"{BASE_URL}/dining/"

print(
    "Dining page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        DINING_URL
    )
)

Dining page allowed: True


In [17]:
time.sleep(3)

dining_response = requests.get(
    DINING_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", dining_response.status_code)
print("Final URL:", dining_response.url)
print("Content type:", dining_response.headers.get("Content-Type"))
print("HTML characters:", len(dining_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/dining/
Content type: text/html; charset=UTF-8
HTML characters: 998709


## Dining Page Structure Inspection

The dining page returned a successful response. The page headings will be inspected to identify possible restaurants, bars, cafés, and unrelated website content.

In [18]:
dining_soup = BeautifulSoup(
    dining_response.text,
    "lxml"
)

dining_page_title = (
    dining_soup.title.get_text(strip=True)
    if dining_soup.title
    else "No title found"
)

print("Page title:", dining_page_title)

Page title: DINING - Taal Vista Hotel


In [19]:
dining_headings = []

for heading in dining_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text and heading_text not in dining_headings:
        dining_headings.append(heading_text)

print(
    "Unique headings found:",
    len(dining_headings)
)
print()

for heading_text in dining_headings[:50]:
    print(heading_text)

Unique headings found: 12

Dining
Broad and Open
VERANDA
Sustainable and Natural
TĀZA FRESH TABLE
Grand and Iconic
LOBBY LOUNGE
pasalubong
alta ridge bar
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


In [20]:
for heading in dining_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text:
        print("Heading:", heading_text)
        print("Tag:", heading.name)
        print("Class:", heading.get("class"))
        print("Parent class:", heading.parent.get("class"))
        print("=" * 50)

Heading: Dining
Tag: h3
Class: []
Parent class: ['h-heading__outer', 'style-5812', 'style-local-116-c8']
Heading: Broad and Open
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c32']
Heading: VERANDA
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c33']
Heading: Sustainable and Natural
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c46']
Heading: TĀZA FRESH TABLE
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c47']
Heading: Grand and Iconic
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c77']
Heading: LOBBY LOUNGE
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c78']
Heading: pasalubong
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c90']
Heading: alta ridge bar
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-1

In [21]:
sample_outlet_names = [
    "VERANDA",
    "pasalubong"
]

for sample_outlet_name in sample_outlet_names:
    sample_heading = dining_soup.find(
        "h2",
        string=lambda text: (
            text
            and text.strip() == sample_outlet_name
        )
    )

    print("OUTLET:", sample_outlet_name)

    for level, ancestor in enumerate(
        sample_heading.parents
    ):
        if level >= 7:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:500])

    print()
    print("=" * 70)

OUTLET: VERANDA

Ancestor level: 0
Tag: div
Class: ['h-heading__outer', 'style-401', 'style-local-116-c33']
Text preview: VERANDA

Ancestor level: 1
Tag: div
Class: ['h-global-transition-all', 'h-heading', 'style-401', 'style-local-116-c33', 'position-relative', 'h-element']
Text preview: VERANDA

Ancestor level: 2
Tag: div
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-100', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: Broad and Open VERANDA For a sentimental and harmonious vibe, partake in our wide selection of international and local cuisine. Served à la carte or buffet, have a sumptuous feast while appreciating cultural performances that include music and dancing. Carrying on the theme of traditional Filipino dishes, the Heirloom section of our menu has always been a family favorite. For a more exciting and interactive setting for the guests, enjoy the buffet section with a live cooking station. Veranda is 

In [22]:
outlet_headings = dining_soup.select(
    "div.h-heading__outer.style-401 h2"
)

print("Outlet headings found:", len(outlet_headings))
print()

for outlet_heading in outlet_headings:
    outlet_name = outlet_heading.get_text(
        " ",
        strip=True
    )

    outlet_container = outlet_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    print("OUTLET:", outlet_name)

    for element in outlet_container.find_all(
        ["h4", "p", "a"]
    ):
        element_text = element.get_text(
            " ",
            strip=True
        )

        if element_text:
            print(
                element.name,
                "|",
                element_text,
                "|",
                element.get("href")
            )

    print("=" * 70)

Outlet headings found: 5

OUTLET: VERANDA
h4 | Broad and Open | None
p | For a sentimental and harmonious vibe, partake in our wide selection of international and local cuisine. Served à la carte or buffet, have a sumptuous feast while appreciating cultural performances that include music and dancing. | None
p | Carrying on the theme of traditional Filipino dishes, the Heirloom section of our menu has always been a family favorite. For a more exciting and interactive setting for the guests, enjoy the buffet section with a live cooking station. | None
p | Veranda is a Green Choice Philippines awardee. | None
p | OPERATING HOURS: | None
p | Open daily, 6:00 AM – 10:00 PM | None
p | NOTE: Schedule for buffet may change without prior notice. | None
a | reserve a table | https://forms.office.com/r/LTwLn6BxuU
a | Menu | https://www.taalvistahotel.com/veranda_menu/
OUTLET: TĀZA FRESH TABLE
h4 | Sustainable and Natural | None
p | For a blissful and breezy surrounding, you can find it at TĀZA F

## Initial Dining Record Extraction

The outlet container includes the outlet name, optional tagline, description, operating schedule, optional schedule note, menu link, and optional reservation link.

The raw extraction will preserve the published wording. Schedule inconsistencies and missing values will be documented during data cleaning rather than corrected during collection.

In [23]:
dining_records = []

for outlet_heading in outlet_headings:
    outlet_name = outlet_heading.get_text(
        " ",
        strip=True
    )

    outlet_container = outlet_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    tagline_element = outlet_container.find("h4")

    if tagline_element:
        tagline = tagline_element.get_text(
            " ",
            strip=True
        )
    else:
        tagline = None

    paragraph_texts = []

    for paragraph in outlet_container.find_all("p"):
        paragraph_text = paragraph.get_text(
            " ",
            strip=True
        )

        if paragraph_text:
            paragraph_texts.append(paragraph_text)

    hours_position = None

    for position, paragraph_text in enumerate(
        paragraph_texts
    ):
        if paragraph_text.upper().startswith(
            "OPERATING HOURS"
        ):
            hours_position = position
            break

    if hours_position is not None:
        description_parts = paragraph_texts[
            :hours_position
        ]
        schedule_parts = paragraph_texts[
            hours_position + 1:
        ]
    else:
        description_parts = paragraph_texts
        schedule_parts = []

    schedule_notes = []
    operating_hours_parts = []

    for schedule_text in schedule_parts:
        if schedule_text.upper().startswith("NOTE"):
            schedule_notes.append(schedule_text)
        else:
            operating_hours_parts.append(
                schedule_text
            )

    menu_url = None
    reservation_url = None

    for link in outlet_container.find_all(
        "a",
        href=True
    ):
        link_text = link.get_text(
            " ",
            strip=True
        ).lower()

        if "menu" in link_text:
            menu_url = link["href"]

        if "reserve" in link_text:
            reservation_url = link["href"]

    dining_records.append({
        "outlet_name": outlet_name,
        "tagline": tagline,
        "outlet_description": " ".join(
            description_parts
        ),
        "operating_hours": " | ".join(
            operating_hours_parts
        ),
        "schedule_note": " ".join(
            schedule_notes
        ) or None,
        "menu_url": menu_url,
        "reservation_url": reservation_url,
        "source_url": DINING_URL,
        "date_collected": date.today().isoformat()
    })

dining_raw_df = pd.DataFrame(dining_records)

dining_raw_df.insert(
    0,
    "outlet_record_id",
    range(1, len(dining_raw_df) + 1)
)

print("Rows:", dining_raw_df.shape[0])
print("Columns:", dining_raw_df.shape[1])
print()
print("Missing values:")
print(dining_raw_df.isnull().sum())

display(dining_raw_df)

Rows: 5
Columns: 10

Missing values:
outlet_record_id      0
outlet_name           0
tagline               2
outlet_description    0
operating_hours       0
schedule_note         4
menu_url              0
reservation_url       3
source_url            0
date_collected        0
dtype: int64


,outlet_record_id,outlet_name,tagline,outlet_description,operating_hours,schedule_note,menu_url,reservation_url,source_url,date_collected
0,1,VERANDA,Broad and Open,"For a sentimental and harmonious vibe, partake...","Open daily, 6:00 AM – 10:00 PM",NOTE: Schedule for buffet may change without p...,https://www.taalvistahotel.com/veranda_menu/,https://forms.office.com/r/LTwLn6BxuU,https://www.taalvistahotel.com/dining/,2026-09-02
1,2,TĀZA FRESH TABLE,Sustainable and Natural,"For a blissful and breezy surrounding, you can...","Open daily, 12:00 NN – 10:00 PM",NaN,https://www.taalvistahotel.com/taza_menu,https://forms.office.com/r/LTwLn6BxuU,https://www.taalvistahotel.com/dining/,2026-09-02
2,3,LOBBY LOUNGE,Grand and Iconic,Underneath the remarkable Tudor-designed rooft...,Sunday to Thursday 10:00 AM – 11:00 PM | Frida...,NaN,https://www.taalvistahotel.com/lobby-lounge-menu/,NaN,https://www.taalvistahotel.com/dining/,2026-09-02
3,4,pasalubong,NaN,"For a taste of sweetness, stop by the hotel’s ...",Sunday to Thursday 10:00 AM – 6:00 PM | Friday...,NaN,https://www.taalvistahotel.com/wp-content/uplo...,NaN,https://www.taalvistahotel.com/dining/,2026-09-02
4,5,alta ridge bar,NaN,"Welcome to ALTA Ridge Bar, your perfect spot t...",Friday – Sunday 4:00 PM – 10:00 PM,NaN,https://www.taalvistahotel.com/alta-ridge-bar-...,NaN,https://www.taalvistahotel.com/dining/,2026-09-02


## Raw Dining Data Quality Check

All required dining fields were extracted successfully. Missing optional fields represent information that was not published for every outlet and will not be filled during raw data collection.

The operating schedule for Pasalubong contains overlapping Sunday references. This will be recorded as a source inconsistency during data cleaning.

In [24]:
required_dining_columns = [
    "outlet_name",
    "outlet_description",
    "operating_hours",
    "menu_url",
    "source_url",
    "date_collected"
]

print(
    "Exact duplicate records:",
    dining_raw_df.duplicated().sum()
)

print()
print("Missing required values:")
print(
    dining_raw_df[
        required_dining_columns
    ].isnull().sum()
)

print()
print("Unique outlet names:")
print(
    dining_raw_df["outlet_name"].nunique()
)

Exact duplicate records: 0

Missing required values:
outlet_name           0
outlet_description    0
operating_hours       0
menu_url              0
source_url            0
date_collected        0
dtype: int64

Unique outlet names:
5


In [25]:
dining_file = (
    raw_data_folder
    / "taal_vista_dining_raw.csv"
)

dining_raw_df.to_csv(
    dining_file,
    index=False
)

dining_verification_df = pd.read_csv(
    dining_file
)

print("File saved:", dining_file)
print(
    "Loaded shape:",
    dining_verification_df.shape
)
print("File exists:", dining_file.exists())
print(
    "Duplicate records:",
    dining_verification_df.duplicated().sum()
)

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_dining_raw.csv
Loaded shape: (5, 10)
File exists: True
Duplicate records: 0


# Event Venue Data Collection

## Purpose

This section collects publicly available information about Taal Vista Hotel’s event venues.

Potential fields include venue name, dimensions, ceiling height, floor area, and capacities for different event arrangements.

In [26]:
EVENTS_URL = f"{BASE_URL}/events/"

print(
    "Events page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        EVENTS_URL
    )
)

Events page allowed: True


In [27]:
time.sleep(3)

events_response = requests.get(
    EVENTS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", events_response.status_code)
print("Final URL:", events_response.url)
print("Content type:", events_response.headers.get("Content-Type"))
print("HTML characters:", len(events_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/events/
Content type: text/html; charset=UTF-8
HTML characters: 1008067


## Event Table Inspection

The events page contains structured venue tables. Each table will be inspected before selecting or combining data because the page may contain different venue groups, layouts, or repeated responsive versions.

In [28]:
from io import StringIO

event_tables = pd.read_html(
    StringIO(events_response.text)
)

print("HTML tables found:", len(event_tables))

HTML tables found: 4


In [29]:
for table_number, event_table in enumerate(
    event_tables,
    start=1
):
    print()
    print("TABLE:", table_number)
    print("Shape:", event_table.shape)
    print("Columns:")
    print(event_table.columns.tolist())

    display(event_table.head(10))


TABLE: 1
Shape: (17, 10)
Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400
3,BALLROOM 1,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
4,BALLROOM 2,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
5,BALLROOM 3,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
6,COVERED TERRACE,37.90 X 9.60,3.90,363.84,300,NaN,NaN,NaN,220,450
7,CAMIA,8.50 X 6.80,3.40,57.80,50,32,22,22,30,60
8,LILY,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60
9,SANTAN,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60



TABLE: 2
Shape: (14, 10)
Columns:
['VENUE', 'DIMENSION (IN METERS)', 'FLOOR AREA (IN SQ. METERS)', 'CONFERENCE CAPACITY', 'THEATER', 'CLASSROOM', 'U-SHAPE', 'BLOCK', 'BANQUET', 'COCKTAIL']


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,360,180.0,90.0,90.0,270,270
1,BALLROOM 1,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
2,BALLROOM 2,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
3,BALLROOM 3,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
4,COVERED TERRACE,37.90 X 9.60,3.90,363.84,100,NaN,NaN,NaN,80,80
5,CAMIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
6,LILY,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
7,SANTAN,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
8,DAHLIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
9,CHAMPACA,7.70 X 6.55,3.25,50.44,20,14.0,10.0,12.0,10,10



TABLE: 3
Shape: (11, 10)
Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.40,420,320,100,120,250,550
3,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.38,NaN,NaN,NaN,NaN,NaN,70
4,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.9960,100,NaN,NaN,NaN,60,120
5,WALING-WALING 1,7.60 X 7.10,3.25,53.96,42,30,20,15,30,60
6,WALING-WALING 2,7.40 X 7.10,3.25,52.54,42,30,20,15,30,60
7,ROSAL 1,9.65 X 7.95,3.20,76.72,60,30,20,15,30,80
8,ROSAL 2,7.60 X 7.95,3.20,60.42,50,30,20,15,30,70
9,GUMAMELA,8.40 X 5.00,3.20,42.00,NaN,NaN,NaN,12,NaN,NaN



TABLE: 4
Shape: (9, 10)
Columns:
['VENUE', 'DIMENSION (IN METERS)', 'FLOOR AREA (IN SQ. METERS)', 'CONFERENCE CAPACITY', 'THEATER', 'CLASSROOM', 'U-SHAPE', 'BLOCK', 'BANQUET', 'COCKTAIL']


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.400,120.0,80.0,30.0,30.0,100.0,100.0
1,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.380,NaN,NaN,NaN,NaN,NaN,30.0
2,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.996,60.0,NaN,NaN,NaN,30.0,30.0
3,WALING-WALING 1,7.60 X 7.10,3.25,53.960,24.0,16.0,13.0,14.0,15.0,15.0
4,WALING-WALING 2,7.40 X 7.10,3.25,52.540,24.0,16.0,13.0,14.0,15.0,15.0
5,ROSAL 1,9.65 X 7.95,3.20,76.720,24.0,16.0,13.0,14.0,15.0,15.0
6,ROSAL 2,7.60 X 7.95,3.20,60.420,24.0,16.0,13.0,14.0,15.0,15.0
7,GUMAMELA,8.40 X 5.00,3.20,42.000,NaN,NaN,NaN,7.0,NaN,NaN
8,JASMIN,7.70 X 5.15,3.50,39.660,16.0,12.0,10.0,10.0,10.0,10.0


In [30]:
display(event_tables[0])
display(event_tables[1])
display(event_tables[2])

,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400
3,BALLROOM 1,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
4,BALLROOM 2,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
5,BALLROOM 3,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
6,COVERED TERRACE,37.90 X 9.60,3.90,363.84,300,NaN,NaN,NaN,220,450
7,CAMIA,8.50 X 6.80,3.40,57.80,50,32,22,22,30,60
8,LILY,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60
9,SANTAN,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,360,180.0,90.0,90.0,270,270
1,BALLROOM 1,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
2,BALLROOM 2,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
3,BALLROOM 3,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
4,COVERED TERRACE,37.90 X 9.60,3.90,363.84,100,NaN,NaN,NaN,80,80
5,CAMIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
6,LILY,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
7,SANTAN,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
8,DAHLIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
9,CHAMPACA,7.70 X 6.55,3.25,50.44,20,14.0,10.0,12.0,10,10


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.40,420,320,100,120,250,550
3,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.38,NaN,NaN,NaN,NaN,NaN,70
4,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.9960,100,NaN,NaN,NaN,60,120
5,WALING-WALING 1,7.60 X 7.10,3.25,53.96,42,30,20,15,30,60
6,WALING-WALING 2,7.40 X 7.10,3.25,52.54,42,30,20,15,30,60
7,ROSAL 1,9.65 X 7.95,3.20,76.72,60,30,20,15,30,80
8,ROSAL 2,7.60 X 7.95,3.20,60.42,50,30,20,15,30,70
9,GUMAMELA,8.40 X 5.00,3.20,42.00,NaN,NaN,NaN,12,NaN,NaN


## Event Table Context Issue

The events page contains two venue groups and two different capacity sets for each group.

The capacity values will not be combined until the surrounding page labels are identified. The differences may represent event categories, layouts, or another operational distinction.

In [31]:
events_soup = BeautifulSoup(
    events_response.text,
    "lxml"
)

html_tables = events_soup.find_all("table")

print("BeautifulSoup tables found:", len(html_tables))
print()

for table_number, html_table in enumerate(
    html_tables,
    start=1
):
    print("TABLE:", table_number)
    print("Table class:", html_table.get("class"))
    print("Table id:", html_table.get("id"))
    print()

    previous_headings = html_table.find_all_previous(
        ["h1", "h2", "h3", "h4", "h5"],
        limit=8
    )

    print("Nearest previous headings:")

    for previous_heading in reversed(
        previous_headings
    ):
        heading_text = previous_heading.get_text(
            " ",
            strip=True
        )

        if heading_text:
            print(
                previous_heading.name,
                "|",
                heading_text
            )

    print("=" * 70)

BeautifulSoup tables found: 4

TABLE: 1
Table class: ['tablepress', 'tablepress-id-1', 'tbody-has-connected-cells']
Table id: tablepress-1

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 2
Table class: ['tablepress', 'tablepress-id-3']
Table id: tablepress-3

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 3
Table class: ['tablepress', 'tablepress-id-4', 'tbody-has-connected-cells']
Table id: tablepress-4

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 4
Table class: ['tablepress', 'tablepress-id-5']
Table id: tablepress-5

Nearest previous h

In [32]:
for table_number, html_table in enumerate(
    html_tables,
    start=1
):
    print("TABLE:", table_number)

    current_element = html_table

    for level in range(1, 7):
        current_element = current_element.parent

        if current_element is None:
            break

        print()
        print("Parent level:", level)
        print("Tag:", current_element.name)
        print("ID:", current_element.get("id"))
        print("Class:", current_element.get("class"))

        previous_sibling = (
            current_element.find_previous_sibling()
        )

        if previous_sibling:
            sibling_text = previous_sibling.get_text(
                " ",
                strip=True
            )

            print(
                "Previous sibling text:",
                sibling_text[:300]
            )
        else:
            print("Previous sibling text: None")

    print()
    print("=" * 70)

TABLE: 1

Parent level: 1
Tag: div
ID: None
Class: []
Previous sibling text: None

Parent level: 2
Tag: div
ID: None
Class: ['style-1996', 'style-local-135-c106', 'h-hide-md', 'h-hide-sm', 'position-relative', 'h-element']
Previous sibling text: 

Parent level: 3
Tag: div
ID: None
Class: ['w-100', 'h-y-container']
Previous sibling text: None

Parent level: 4
Tag: div
ID: lake-wing
Class: ['h-tabs-item-content', 'h-tabs-item', 'h-tabs-content-horizontal', 'h-tabs-content-135-c104', 'style-1617', 'style-local-135-c104', 'position-relative', 'h-element', 'h-tabs-content-active']
Previous sibling text: LAKE WING MOUNTAIN WING

Parent level: 5
Tag: div
ID: None
Class: ['h-tabs', 'h-tabs-horizontal', 'h-tabs--horizontal--stretch-lg', 'h-tabs--horizontal--stretch-md', 'h-tabs--horizontal--full', 'h-tabs', 'h-tabs-horizontal', 'h-tabs--horizontal--stretch-lg', 'h-tabs--horizontal--stretch-md', 'h-tabs--horizontal--full', 'style-1616', 'style-local-135-c103', 'position-relative', 'h-element']
P

## Event Capacity Data Inconsistency

The events page uses separate tabs for Lake Wing and Mountain Wing. Each wing contains a desktop table and a mobile table.

The desktop and mobile versions contain different capacity values for the same venues. Both versions will be preserved as separate raw datasets.

No capacity value will be corrected or selected during data collection. The differences will be quantified and documented during data cleaning and business analysis.

In [33]:
event_table_details = [
    {
        "table_index": 0,
        "hotel_wing": "Lake Wing",
        "display_version": "Desktop",
        "file_name": "taal_vista_events_lake_desktop_raw.csv"
    },
    {
        "table_index": 1,
        "hotel_wing": "Lake Wing",
        "display_version": "Mobile",
        "file_name": "taal_vista_events_lake_mobile_raw.csv"
    },
    {
        "table_index": 2,
        "hotel_wing": "Mountain Wing",
        "display_version": "Desktop",
        "file_name": "taal_vista_events_mountain_desktop_raw.csv"
    },
    {
        "table_index": 3,
        "hotel_wing": "Mountain Wing",
        "display_version": "Mobile",
        "file_name": "taal_vista_events_mountain_mobile_raw.csv"
    }
]

saved_event_files = []

for table_detail in event_table_details:
    event_raw_df = event_tables[
        table_detail["table_index"]
    ].copy()

    event_raw_df["hotel_wing"] = (
        table_detail["hotel_wing"]
    )

    event_raw_df["display_version"] = (
        table_detail["display_version"]
    )

    event_raw_df["source_url"] = EVENTS_URL
    event_raw_df["date_collected"] = (
        date.today().isoformat()
    )

    event_file = (
        raw_data_folder
        / table_detail["file_name"]
    )

    event_raw_df.to_csv(
        event_file,
        index=False
    )

    saved_event_files.append(event_file)

    print("File:", event_file.name)
    print("Shape:", event_raw_df.shape)
    print("Exists:", event_file.exists())
    print()

File: taal_vista_events_lake_desktop_raw.csv
Shape: (17, 14)
Exists: True

File: taal_vista_events_lake_mobile_raw.csv
Shape: (14, 14)
Exists: True

File: taal_vista_events_mountain_desktop_raw.csv
Shape: (11, 14)
Exists: True

File: taal_vista_events_mountain_mobile_raw.csv
Shape: (9, 14)
Exists: True



# Promotions Data Collection

## Purpose

This section collects current publicly presented promotional offers from the official Taal Vista Hotel promotions page.

Potential fields include promotion name, category, description, advertised rate, room category, inclusions, validity information, booking link, source URL, and collection date.

In [34]:
PROMOTIONS_URL = (
    f"{BASE_URL}/family-friendly-tagaytay-hotel-deals/"
)

print(
    "Promotions page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        PROMOTIONS_URL
    )
)

Promotions page allowed: True


In [35]:
time.sleep(3)

promotions_response = requests.get(
    PROMOTIONS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", promotions_response.status_code)
print("Final URL:", promotions_response.url)
print("Content type:", promotions_response.headers.get("Content-Type"))
print("HTML characters:", len(promotions_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/family-friendly-tagaytay-hotel-deals/
Content type: text/html; charset=UTF-8
HTML characters: 1203900


## Promotions Page Structure Inspection

The page headings will be inspected to identify promotion categories, promotion names, and unrelated website elements before defining extraction rules.

In [36]:
promotions_soup = BeautifulSoup(
    promotions_response.text,
    "lxml"
)

promotions_page_title = (
    promotions_soup.title.get_text(strip=True)
    if promotions_soup.title
    else "No title found"
)

print("Page title:", promotions_page_title)


Page title: Promos - Taal Vista Hotel


In [37]:
promotion_headings = []

for heading in promotions_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if (
        heading_text
        and heading_text not in promotion_headings
    ):
        promotion_headings.append(heading_text)

print(
    "Unique headings found:",
    len(promotion_headings)
)
print()

for heading_text in promotion_headings[:100]:
    print(heading_text)

Unique headings found: 29

Exclusive Hotel Packages in Tagaytay City
RAIN, RAIN, GETAWAY!
Mobile exclusive deals
elevated escapes
Suite serenity: A moment of luxury
PLAN AHEAD PAY LESS!
BEST TO BOOK DIRECT!
PLATE FOR THE PLANET
Perfect Pairings: Bites & Beer
DINE UNDER THE STARS
GLAM PICNIC AT THE LAWN
IN-ROOM DINING SPECIALS
CUISINE MEETS CULTURE
GELATO
FILIPINO CLASSICS
GINGER GLAZED RIBS
A WALK THROUGH TIME
GRILL AT ALTA
AFTERNOON TEA TIME
TIMELESS CHRISTMAS FESTIVITIES
EDUXPERIENCE: LEARN & EXPLORE AT TAAL VISTA HOTEL
CATERING-ON-THE-GO
WEDDING PACKAGE
PARTY PACKAGE
A WARM ESCAPE FROM THE RAIN
SOY AND FIRE
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


In [38]:
excluded_promotion_headings = [
    "Exclusive Hotel Packages in Tagaytay City",
    "Privacy Overview",
    "Chinese New Year Offerings at Taal Vista Hotel",
    "Breathe Fresh"
]

for heading in promotions_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if (
        heading_text
        and heading_text
        not in excluded_promotion_headings
    ):
        print("Heading:", heading_text)
        print("Tag:", heading.name)
        print("Class:", heading.get("class"))
        print("Parent tag:", heading.parent.name)
        print("Parent class:", heading.parent.get("class"))
        print("=" * 60)

Heading: RAIN, RAIN, GETAWAY!
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c24']
Heading: Mobile exclusive deals
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c35']
Heading: elevated escapes
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c46']
Heading: Suite serenity: A moment of luxury
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c57']
Heading: PLAN AHEAD PAY LESS!
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c68']
Heading: BEST TO BOOK DIRECT!
Tag: h1
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-4754', 'style-local-136-c79']
Heading: PLATE FOR THE PLANET
Tag: h2
Class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-2415', 'style-local-136-c95']
Heading: Perfect Pairings: Bites & B

In [39]:
promotion_candidates = []

for heading in promotions_soup.find_all(
    ["h1", "h2"]
):
    heading_text = heading.get_text(" ", strip=True)

    if (
        heading_text
        and heading_text
        not in excluded_promotion_headings
    ):
        tab_container = heading.find_parent(
            "div",
            class_="h-tabs-item-content"
        )

        if tab_container:
            tab_id = tab_container.get("id")
            tab_class = tab_container.get("class")
        else:
            tab_id = None
            tab_class = None

        promotion_candidates.append({
            "promotion_name": heading_text,
            "heading_tag": heading.name,
            "heading_parent_class": " ".join(
                heading.parent.get("class", [])
            ),
            "tab_id": tab_id,
            "tab_class": tab_class
        })

promotion_structure_df = pd.DataFrame(
    promotion_candidates
)

display(
    promotion_structure_df[
        [
            "promotion_name",
            "heading_tag",
            "tab_id"
        ]
    ]
)

print()
print("Promotion counts by tab:")
print(
    promotion_structure_df[
        "tab_id"
    ].value_counts(dropna=False)
)

,promotion_name,heading_tag,tab_id
0,"RAIN, RAIN, GETAWAY!",h1,accommodation
1,Mobile exclusive deals,h1,accommodation
2,elevated escapes,h1,accommodation
3,Suite serenity: A moment of luxury,h1,accommodation
4,PLAN AHEAD PAY LESS!,h1,accommodation
5,BEST TO BOOK DIRECT!,h1,accommodation
6,PLATE FOR THE PLANET,h2,dining
7,Perfect Pairings: Bites & Beer,h2,dining
8,DINE UNDER THE STARS,h2,dining
9,GLAM PICNIC AT THE LAWN,h2,dining



Promotion counts by tab:
tab_id
dining           12
accommodation     6
events-2          5
rain-the-spa      2
Name: count, dtype: int64


In [40]:
sample_promotion_names = [
    "RAIN, RAIN, GETAWAY!",
    "PLATE FOR THE PLANET",
    "TIMELESS CHRISTMAS FESTIVITIES",
    "A WARM ESCAPE FROM THE RAIN"
]

for sample_name in sample_promotion_names:
    sample_heading = promotions_soup.find(
        ["h1", "h2"],
        string=lambda text: (
            text
            and text.strip() == sample_name
        )
    )

    promotion_container = sample_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    print("PROMOTION:", sample_name)
    print()

    for element in promotion_container.find_all(
        ["p", "a"]
    ):
        element_text = element.get_text(
            " ",
            strip=True
        )

        if element_text:
            print(
                element.name,
                "|",
                element_text,
                "|",
                element.get("href")
            )

    print("=" * 70)

PROMOTION: RAIN, RAIN, GETAWAY!

p | Escape the rain with a cozy stay, delightful perks, timeless Tagaytay views and an exclusive Brickseum experience. | None
p | Rates start at Php 8,000 for a Superior Room. | None
PROMOTION: PLATE FOR THE PLANET

p | Fresh from Tagaytay, our pineapple-inspired creations support local farmers while bringing bright, local flavors to your plate. | None
p | Rates start at Php 130 per person | None
PROMOTION: TIMELESS CHRISTMAS FESTIVITIES

p | Thoughtfully curated celebrations inspired by timeless traditions, exceptional hospitality, and meaningful connections that bring teams together. | None
PROMOTION: A WARM ESCAPE FROM THE RAIN

p | Turn gloomy weather into a moment of self care | None


In [41]:
sample_promotion_names = [
    "RAIN, RAIN, GETAWAY!",
    "PLATE FOR THE PLANET"
]

for sample_name in sample_promotion_names:
    sample_heading = promotions_soup.find(
        ["h1", "h2"],
        string=lambda text: (
            text
            and text.strip() == sample_name
        )
    )

    print("PROMOTION:", sample_name)

    for level, ancestor in enumerate(
        sample_heading.parents
    ):
        if level >= 8:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("ID:", ancestor.get("id"))
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:700])

    print()
    print("=" * 70)

PROMOTION: RAIN, RAIN, GETAWAY!

Ancestor level: 0
Tag: div
ID: None
Class: ['h-heading__outer', 'style-4754', 'style-local-136-c24']
Text preview: RAIN, RAIN, GETAWAY!

Ancestor level: 1
Tag: div
ID: None
Class: ['h-global-transition-all', 'h-heading', 'style-4754', 'style-local-136-c24', 'position-relative', 'h-element']
Text preview: RAIN, RAIN, GETAWAY!

Ancestor level: 2
Tag: div
ID: None
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-auto', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: RAIN, RAIN, GETAWAY! Escape the rain with a cozy stay, delightful perks, timeless Tagaytay views and an exclusive Brickseum experience. Rates start at Php 8,000 for a Superior Room.

Ancestor level: 3
Tag: div
ID: None
Class: ['d-flex', 'h-flex-basis', 'h-column__inner', 'h-px-lg-1', 'h-px-md-1', 'h-px-1', 'v-inner-lg-1', 'v-inner-md-1', 'v-inner-1', 'style-2413', 'style-local-136-c23', 'position-relative']
Text preview: RAI

In [42]:
sample_promotion_names = [
    "RAIN, RAIN, GETAWAY!",
    "PLATE FOR THE PLANET",
    "TIMELESS CHRISTMAS FESTIVITIES",
    "A WARM ESCAPE FROM THE RAIN"
]

for sample_name in sample_promotion_names:
    sample_heading = promotions_soup.find(
        ["h1", "h2"],
        string=lambda text: (
            text
            and text.strip() == sample_name
        )
    )

    content_container = sample_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    content_strings = list(
        content_container.stripped_strings
    )

    link_container = None

    for ancestor in sample_heading.parents:
        links = ancestor.find_all(
            "a",
            href=True
        )

        if links:
            link_container = ancestor
            break

    print("PROMOTION:", sample_name)
    print("Content strings:", content_strings)
    print("Links:")

    if link_container:
        for link in link_container.find_all(
            "a",
            href=True
        ):
            print(
                link.get_text(" ", strip=True),
                "|",
                link["href"]
            )
    else:
        print("No links found")

    print("=" * 70)

PROMOTION: RAIN, RAIN, GETAWAY!
Content strings: ['RAIN, RAIN, GETAWAY!', 'Escape the rain with a cozy stay, delightful perks, timeless Tagaytay views and an exclusive Brickseum experience.', 'Rates start at Php 8,000 for a Superior Room.']
Links:
KNOW MORE | https://www.taalvistahotel.com/wp-content/uploads/2026/07/Rain-Rain-Getaway.jpg
BOOK NOW | https://www.simplebooking.it/ibe2/hotel/6757?lang=EN&cur=PHP&guests=A%2CA&in=2026-09-12&out=2026-09-13
PROMOTION: PLATE FOR THE PLANET
Content strings: ['PLATE FOR THE PLANET', 'Fresh from Tagaytay, our pineapple-inspired creations support local farmers while bringing bright, local flavors to your plate.', 'Rates start at Php 130 per person']
Links:
KNOW MORE | https://www.taalvistahotel.com/wp-content/uploads/2026/07/Plate-for-the-Planet-for-Website.jpg
PROMOTION: TIMELESS CHRISTMAS FESTIVITIES
Content strings: ['TIMELESS CHRISTMAS FESTIVITIES', 'Thoughtfully curated celebrations inspired by timeless traditions, exceptional hospitality, and

## Initial Promotion Record Extraction

Each promotion will retain its published category, name, descriptive text, detail link, optional action label, and optional action URL.

Descriptions and advertised rates will remain together in the raw `promotion_details` field. They will be separated during data cleaning because their wording is not consistent across all promotions.

In [44]:
category_mapping = {
    "accommodation": "Accommodation",
    "dining": "Dining",
    "events-2": "Events",
    "rain-the-spa": "Rain, The Spa"
}

promotion_records = []
unmatched_promotions = []

for _, promotion_row in (
    promotion_structure_df.iterrows()
):
    promotion_name = promotion_row[
        "promotion_name"
    ]

    promotion_heading = None

    for candidate_heading in (
        promotions_soup.find_all(["h1", "h2"])
    ):
        candidate_text = candidate_heading.get_text(
            " ",
            strip=True
        )

        if candidate_text == promotion_name:
            promotion_heading = candidate_heading
            break

    if promotion_heading is None:
        unmatched_promotions.append(
            promotion_name
        )
        continue

    content_container = promotion_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    content_strings = list(
        content_container.stripped_strings
    )

    promotion_details = " | ".join(
        content_strings[1:]
    )

    promotion_links = []

    for ancestor in promotion_heading.parents:
        links = ancestor.find_all(
            "a",
            href=True
        )

        if links:
            for link in links:
                link_text = link.get_text(
                    " ",
                    strip=True
                )
                link_url = link["href"]

                link_pair = (
                    link_text,
                    link_url
                )

                if link_pair not in promotion_links:
                    promotion_links.append(
                        link_pair
                    )

            break

    details_url = None
    action_labels = []
    action_urls = []

    for link_text, link_url in promotion_links:
        if link_text.upper() == "KNOW MORE":
            details_url = link_url
        else:
            action_labels.append(link_text)
            action_urls.append(link_url)

    promotion_records.append({
        "promotion_category": category_mapping.get(
            promotion_row["tab_id"],
            promotion_row["tab_id"]
        ),
        "promotion_name": promotion_name,
        "promotion_details": promotion_details,
        "details_url": details_url,
        "action_label": " | ".join(
            action_labels
        ) or None,
        "action_url": " | ".join(
            action_urls
        ) or None,
        "source_url": PROMOTIONS_URL,
        "date_collected": date.today().isoformat()
    })

promotions_raw_df = pd.DataFrame(
    promotion_records
)

promotions_raw_df.insert(
    0,
    "promotion_record_id",
    range(1, len(promotions_raw_df) + 1)
)

print(
    "Unmatched promotions:",
    unmatched_promotions
)
print("Rows:", promotions_raw_df.shape[0])
print("Columns:", promotions_raw_df.shape[1])
print()
print("Missing values:")
print(promotions_raw_df.isnull().sum())
print()
print("Promotions by category:")
print(
    promotions_raw_df[
        "promotion_category"
    ].value_counts()
)

display(promotions_raw_df)

Unmatched promotions: []
Rows: 25
Columns: 9

Missing values:
promotion_record_id    0
promotion_category     0
promotion_name         0
promotion_details      0
details_url            0
action_label           9
action_url             9
source_url             0
date_collected         0
dtype: int64

Promotions by category:
promotion_category
Dining           12
Accommodation     6
Events            5
Rain, The Spa     2
Name: count, dtype: int64


,promotion_record_id,promotion_category,promotion_name,promotion_details,details_url,action_label,action_url,source_url,date_collected
0,1,Accommodation,"RAIN, RAIN, GETAWAY!","Escape the rain with a cozy stay, delightful p...",https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757?l...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
1,2,Accommodation,Mobile exclusive deals,Your dream staycation is just a tap away. Book...,https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757?l...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
2,3,Accommodation,elevated escapes,Where comfort meets calm and every view feels ...,https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757/?...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
3,4,Accommodation,Suite serenity: A moment of luxury,Surrender to an atmosphere of stillness and su...,https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757/?...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
4,5,Accommodation,PLAN AHEAD PAY LESS!,Plan you Tagaytay escape ahead and enjoy 30% o...,https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757/?...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
5,6,Accommodation,BEST TO BOOK DIRECT!,"Book Direct, Save More! | Rates start at Php 8...",https://www.taalvistahotel.com/wp-content/uplo...,BOOK NOW,https://www.simplebooking.it/ibe2/hotel/6757/?...,https://www.taalvistahotel.com/family-friendly...,2026-09-02
6,7,Dining,PLATE FOR THE PLANET,"Fresh from Tagaytay, our pineapple-inspired cr...",https://www.taalvistahotel.com/wp-content/uplo...,NaN,NaN,https://www.taalvistahotel.com/family-friendly...,2026-09-02
7,8,Dining,Perfect Pairings: Bites & Beer,Enjoy the perfect pairing of crowd-favorite sa...,https://www.taalvistahotel.com/wp-content/uplo...,NaN,NaN,https://www.taalvistahotel.com/family-friendly...,2026-09-02
8,9,Dining,DINE UNDER THE STARS,"Feel the love in the most romantic setting, pe...",,BOOK YOUR TABLE,https://bit.ly/TVHDiningReservations,https://www.taalvistahotel.com/family-friendly...,2026-09-02
9,10,Dining,GLAM PICNIC AT THE LAWN,"Spend worthwhile afternoons with good food, go...",,RESERVE NOW,https://forms.office.com/r/LTwLn6BxuU,https://www.taalvistahotel.com/family-friendly...,2026-09-02


## Promotion Description Correction

One promotion heading contains multiple nested text elements. The initial extraction removed only the first heading fragment, causing the remaining fragment to appear in the promotion details.

The details will be reconstructed by excluding every text node located inside an `h1` or `h2` element.

In [45]:
for row_index, promotion_row in (
    promotions_raw_df.iterrows()
):
    promotion_name = promotion_row[
        "promotion_name"
    ]

    promotion_heading = None

    for candidate_heading in (
        promotions_soup.find_all(["h1", "h2"])
    ):
        candidate_text = candidate_heading.get_text(
            " ",
            strip=True
        )

        if candidate_text == promotion_name:
            promotion_heading = candidate_heading
            break

    content_container = promotion_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    detail_strings = []

    for text_node in content_container.find_all(
        string=True
    ):
        clean_text = text_node.strip()

        if (
            clean_text
            and text_node.find_parent(
                ["h1", "h2"]
            ) is None
        ):
            detail_strings.append(clean_text)

    promotions_raw_df.loc[
        row_index,
        "promotion_details"
    ] = " | ".join(detail_strings)

eduxperience_check = promotions_raw_df[
    promotions_raw_df["promotion_name"].str.contains(
        "EDUXPERIENCE",
        case=False,
        na=False
    )
]

display(
    eduxperience_check[
        [
            "promotion_name",
            "promotion_details"
        ]
    ]
)

,promotion_name,promotion_details
19,EDUXPERIENCE: LEARN & EXPLORE AT TAAL VISTA HOTEL,Enjoy a unique educational experience with ins...


In [46]:
promotion_quality_df = (
    promotions_raw_df.replace(
        r"^\s*$",
        pd.NA,
        regex=True
    )
)

required_promotion_columns = [
    "promotion_category",
    "promotion_name",
    "promotion_details",
    "source_url",
    "date_collected"
]

print("Rows:", promotion_quality_df.shape[0])
print("Columns:", promotion_quality_df.shape[1])
print()
print(
    "Exact duplicate records:",
    promotion_quality_df.duplicated().sum()
)
print(
    "Repeated promotion names:",
    promotion_quality_df[
        "promotion_name"
    ].str.lower().duplicated().sum()
)
print()
print("Missing required values:")
print(
    promotion_quality_df[
        required_promotion_columns
    ].isnull().sum()
)
print()
print("Missing optional values:")
print(
    promotion_quality_df[
        [
            "details_url",
            "action_label",
            "action_url"
        ]
    ].isnull().sum()
)

Rows: 25
Columns: 9

Exact duplicate records: 0
Repeated promotion names: 0

Missing required values:
promotion_category    0
promotion_name        0
promotion_details     0
source_url            0
date_collected        0
dtype: int64

Missing optional values:
details_url     11
action_label     9
action_url       9
dtype: int64


In [47]:
promotions_file = (
    raw_data_folder
    / "taal_vista_promotions_raw.csv"
)

promotion_quality_df.to_csv(
    promotions_file,
    index=False
)

promotions_verification_df = pd.read_csv(
    promotions_file
)

print("File saved:", promotions_file)
print(
    "Loaded shape:",
    promotions_verification_df.shape
)
print("File exists:", promotions_file.exists())
print(
    "Duplicate records:",
    promotions_verification_df.duplicated().sum()
)

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_promotions_raw.csv
Loaded shape: (25, 9)
File exists: True
Duplicate records: 0


# Facilities Data Collection

## Purpose

This section collects the publicly presented hotel facilities from the official facilities page.

Potential fields include facility name, description, operating information, access conditions, booking link, source URL, and collection date.

In [48]:
FACILITIES_URL = f"{BASE_URL}/facilities/"

print(
    "Facilities page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        FACILITIES_URL
    )
)

Facilities page allowed: True


In [49]:
time.sleep(3)

facilities_response = requests.get(
    FACILITIES_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", facilities_response.status_code)
print("Final URL:", facilities_response.url)
print("Content type:", facilities_response.headers.get("Content-Type"))
print("HTML characters:", len(facilities_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/facilities/
Content type: text/html; charset=UTF-8
HTML characters: 952583


## Facilities Page Structure Inspection

The page headings will be inspected to identify facility names and separate them from general page content, privacy sections, and popup information.

In [50]:
facilities_soup = BeautifulSoup(
    facilities_response.text,
    "lxml"
)

facilities_page_title = (
    facilities_soup.title.get_text(strip=True)
    if facilities_soup.title
    else "No title found"
)

print("Page title:", facilities_page_title)

Page title: FACILITIES - Taal Vista Hotel


In [51]:
facility_headings = []

for heading in facilities_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if (
        heading_text
        and heading_text not in facility_headings
    ):
        facility_headings.append(heading_text)

print(
    "Unique headings found:",
    len(facility_headings)
)
print()

for heading_text in facility_headings[:50]:
    print(heading_text)

Unique headings found: 4

Complete Tagaytay Hotel Facilities in Taal Vista Hotel
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


In [52]:
expected_facility_names = [
    "KIDS' CORNER AND GAME ROOM",
    "RAIN, THE SPA",
    "FITNESS CENTER",
    "SWIMMING POOL",
    "KULTURA BUTIK"
]

for facility_name in expected_facility_names:
    matching_text_nodes = facilities_soup.find_all(
        string=lambda text: (
            text
            and facility_name.lower()
            in text.strip().lower()
        )
    )

    print("FACILITY:", facility_name)
    print(
        "Matching text nodes:",
        len(matching_text_nodes)
    )

    for text_node in matching_text_nodes[:5]:
        print(
            "Text:",
            text_node.strip()
        )
        print(
            "Parent tag:",
            text_node.parent.name
        )
        print(
            "Parent class:",
            text_node.parent.get("class")
        )

    print("=" * 60)

FACILITY: KIDS' CORNER AND GAME ROOM
Matching text nodes: 0
FACILITY: RAIN, THE SPA
Matching text nodes: 5
Text: RAIN, THE SPA
Parent tag: a
Parent class: None
Text: RAIN, THE SPA
Parent tag: a
Parent class: None
Text: Taal Vista Hotel stands out above the rest among other properties in Tagaytay. Hotel facilities include a large outdoor swimming pool hidden from view to provide privacy. To help you give an energy boost or complete your fitness regimen, there’s an onsite gym. Looking for some entertainment? Tagaytay Hotel Vista has its own kid’s corner and games room. For complete relaxation, Rain, The Spa offers rejuvenating body treatments. Before leaving Tagaytay, check the hotel Cake Shop or Kultura Butik for some souvenirs. Taal Vista Hotel has hotel facilities other Tagaytay properties don’t have.
Parent tag: span
Parent class: None
Text: RAIN, THE SPA
Parent tag: h5
Parent class: []
Text: Nestled in the cool comforts of Tagaytay, Rain, The Spa offers a serene escape where relaxat

In [53]:
facility_h5_headings = (
    facilities_soup.find_all("h5")
)

print(
    "H5 headings found:",
    len(facility_h5_headings)
)
print()

for facility_heading in facility_h5_headings:
    facility_name = facility_heading.get_text(
        " ",
        strip=True
    )

    facility_container = facility_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    paragraph_texts = []

    if facility_container:
        for paragraph in facility_container.find_all(
            "p"
        ):
            paragraph_text = paragraph.get_text(
                " ",
                strip=True
            )

            if paragraph_text:
                paragraph_texts.append(
                    paragraph_text
                )

    print("Facility:", facility_name)
    print("Paragraphs:", paragraph_texts)
    print("=" * 70)

H5 headings found: 5

Facility: KIDS’ CORNER AND GAME ROOM
Paragraphs: ['Happy place your kids will love – with colorful toys, mini ball pit and slide plus kids’ activities.', 'Operating hours:', 'Open\xa0daily from 8AM to 8PM']
Facility: RAIN, THE SPA
Paragraphs: ['Nestled in the cool comforts of Tagaytay, Rain, The Spa offers a serene escape where relaxation flows naturally and every moment is unhurried.', 'Operating hours:', 'Sunday – Thursday: 10 AM to 10 PM', 'Friday – Saturday: 10 AM to 12 MN']
Facility: FITNESS CENTER
Paragraphs: ['A variety of exercise equipment is available for complimentary use.', 'Operating hours:', 'Open\xa0daily from 6AM to 9PM']
Facility: SWIMMING POOL
Paragraphs: ['Take a dip or swim some laps in our pool located in the Mountain Wings of Taal Vista Hotel in Tagaytay City. Nothing feels better than a cool morning or afternoon swim.', 'Operating hours:', 'Open daily from 7AM to 6PM']
Facility: KULTURA BUTIK
Paragraphs: ['Bring a bit of Tagaytay to your lov

In [54]:
for facility_heading in facility_h5_headings:
    facility_name = facility_heading.get_text(
        " ",
        strip=True
    )

    facility_container = facility_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    facility_links = facility_container.find_all(
        "a",
        href=True
    )

    print("FACILITY:", facility_name)

    if facility_links:
        for link in facility_links:
            print(
                link.get_text(" ", strip=True),
                "|",
                link["href"]
            )
    else:
        print("No facility link found")

    print("=" * 60)

FACILITY: KIDS’ CORNER AND GAME ROOM
No facility link found
FACILITY: RAIN, THE SPA
No facility link found
FACILITY: FITNESS CENTER
No facility link found
FACILITY: SWIMMING POOL
No facility link found
FACILITY: KULTURA BUTIK
No facility link found


## Initial Facility Record Extraction

The facilities page contains five facility cards. Each card provides a facility name, descriptive information, and operating schedule.

No facility specific links are included in the cards. The facilities page URL will therefore serve as the source URL for every record.

In [55]:
facility_records = []

for facility_heading in facility_h5_headings:
    facility_name = facility_heading.get_text(
        " ",
        strip=True
    )

    facility_container = facility_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    paragraph_texts = []

    for paragraph in facility_container.find_all("p"):
        paragraph_text = paragraph.get_text(
            " ",
            strip=True
        )

        if paragraph_text:
            paragraph_texts.append(paragraph_text)

    hours_position = None

    for position, paragraph_text in enumerate(
        paragraph_texts
    ):
        if paragraph_text.lower().startswith(
            "operating hours"
        ):
            hours_position = position
            break

    if hours_position is not None:
        description_parts = paragraph_texts[
            :hours_position
        ]
        hours_parts = paragraph_texts[
            hours_position + 1:
        ]
    else:
        description_parts = paragraph_texts
        hours_parts = []

    facility_records.append({
        "facility_name": facility_name,
        "facility_description": " ".join(
            description_parts
        ),
        "operating_hours": " | ".join(
            hours_parts
        ) or None,
        "source_url": FACILITIES_URL,
        "date_collected": date.today().isoformat()
    })

facilities_raw_df = pd.DataFrame(
    facility_records
)

facilities_raw_df.insert(
    0,
    "facility_record_id",
    range(1, len(facilities_raw_df) + 1)
)

print("Rows:", facilities_raw_df.shape[0])
print("Columns:", facilities_raw_df.shape[1])
print()
print("Missing values:")
print(facilities_raw_df.isnull().sum())
print()
print(
    "Exact duplicate records:",
    facilities_raw_df.duplicated().sum()
)

display(facilities_raw_df)

Rows: 5
Columns: 6

Missing values:
facility_record_id      0
facility_name           0
facility_description    0
operating_hours         0
source_url              0
date_collected          0
dtype: int64

Exact duplicate records: 0


,facility_record_id,facility_name,facility_description,operating_hours,source_url,date_collected
0,1,KIDS’ CORNER AND GAME ROOM,Happy place your kids will love – with colorfu...,Open daily from 8AM to 8PM,https://www.taalvistahotel.com/facilities/,2026-09-02
1,2,"RAIN, THE SPA","Nestled in the cool comforts of Tagaytay, Rain...",Sunday – Thursday: 10 AM to 10 PM | Friday – S...,https://www.taalvistahotel.com/facilities/,2026-09-02
2,3,FITNESS CENTER,A variety of exercise equipment is available f...,Open daily from 6AM to 9PM,https://www.taalvistahotel.com/facilities/,2026-09-02
3,4,SWIMMING POOL,Take a dip or swim some laps in our pool locat...,Open daily from 7AM to 6PM,https://www.taalvistahotel.com/facilities/,2026-09-02
4,5,KULTURA BUTIK,Bring a bit of Tagaytay to your loved ones bac...,Open Thursday to Monday from 9AM to 6PM,https://www.taalvistahotel.com/facilities/,2026-09-02
